# Lilly — Run B: English → Bosnian, back-translation from MaCoCu-bs (CC0)

Pre-registered in `training/PREREGISTRATION.md`, "Run B — reply (English → Bosnian)
— back-translation from MaCoCu-bs". This notebook **is** that run, launch-ready.

Run B changes exactly one thing versus the shipped en→bs adapter: the training
**data**. It back-translates real Bosnian from MaCoCu-bs (730M words, CC0) into
synthetic English with the **shipped bs→en forward build**, pairs the synthetic
English source with the real Bosnian target, and mixes it 1:1 with the existing
en→bs parallel. Everything else — base, LoRA rank, seed, epochs, `>>bos_Latn<<`
tagging, the eval path — is held identical.

The four bars are locked in the pre-registration and are scored **at home on the
served path**, not here: chrF2 > 61.55 (CI excludes 0), BLEU CI-low ≥ −0.30, form
rate ≥ 99.0%, label gap ≥ +15. All must hold. Launch is the owner's call:
`python3 scripts/kaggle_train.py translation-en-bs-backtrans`.

In [ ]:
# 0. RUN B — the reply LoRA, fixed. The one variable is the training data.
# Set here, in the committed file, so the notebook that produced a build is the
# notebook in git (the version-skew that once killed a run at In [6]).
DIRECTION = "en-bs"
ARM = "lora"
assert DIRECTION == "en-bs" and ARM == "lora", \
    "Run B is the reply LoRA; it does not train the other arm or direction"

# The bs->en forward build that does the back-translation MUST be the shipped
# build every published served-path number was scored on (training/RESULTS-product.md).
# Identified by content fingerprint, checked before a single sentence is
# translated -- a different forward model makes different synthetic English and a
# different experiment.
FORWARD_FINGERPRINT = "1aedcc11231cdf50817ff12f99ff0d1e"

# How many MaCoCu-bs sentences to back-translate. The pre-registration names 1.0M
# kept pairs, 1:1 synthetic:real. Back-translating 1M sentences with a large model
# AND training on ~2M examples does not fit one 12h T4 session, so this run FAILS
# LOUD if generation cannot finish inside MAX_BT_SECONDS rather than quietly
# shipping a smaller run: reduce BACKTRANS_N, or split the generation into its own
# session (owner's call). It never under-produces silently.
BACKTRANS_N = 1_000_000
MAX_BT_SECONDS = 9 * 3600     # headroom under the 12h wall for training + scoring

import os
os.environ["LILLY_RUN_ID"] = "backtrans-en-bs"

# en-bs paths. Getting one wrong produces a model that translates the wrong way,
# or an adapter zipped under another name and installed over the shipped one.
PATHS = {"adapter": "models/lilly/adapter-en-bs",
         "results": "training/RESULTS-en-bs.md",
         "zip": "lilly-adapter-en-bs-backtrans", "tag": ">>bos_Latn<<"}
print("Run B — direction:", DIRECTION, "| arm:", ARM, "| N:", BACKTRANS_N)

In [ ]:
# 1. Stop here unless the machine is actually set up
# Kaggle marks a version "complete" whenever no cell RAISES — a shell command that
# fails is not enough. So every check below is Python, and every later step is a
# checked subprocess. A misconfigured run should die in ten seconds, not in ten hours.
import os, subprocess, sys, urllib.error, urllib.request
from pathlib import Path

import torch
assert torch.cuda.is_available(), (
    "No GPU. Right panel -> Session options -> Accelerator -> GPU, then Save & Run All again.")

# Pin to one GPU on purpose. With two visible, the trainer splits each batch across
# both, which doubles the effective batch and halves the optimizer steps while the
# learning rate stays put — a different recipe than the one these numbers were set for.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(torch.cuda.device_count(), "GPU(s) visible, using:", torch.cuda.get_device_name(0))

def reachable(url):
    try:
        urllib.request.urlopen(url, timeout=20).close()
    except urllib.error.HTTPError:
        pass          # a status code still proves we got out
    except Exception as exc:
        raise SystemExit(
            f"Cannot reach {url} ({exc}). Right panel -> Session options -> Internet -> On.")

for host in ("https://github.com", "https://pypi.org", "https://object.pouta.csc.fi"):
    reachable(host)
print("network ok")

# Under /kaggle/working, so it is Output and outlives the log.
TEE = Path("/kaggle/working/stdout.txt")

def run(*cmd, quiet=False):
    """Run a child process and put its output somewhere it can be found.

    Kaggle's log holds what this notebook process prints. A child process
    writing to its own stdout is not in it: subprocess.run(check=True) can
    COMPLETE a kernel that never printed a loss line (docs/kaggle-fail-stop.md,
    item 15). Read the child's output here, reprint it, and tee it into
    Output so the numbers survive a dropped log too.
    """
    line = "$ " + " ".join(str(c) for c in cmd)
    print(line, flush=True)
    with TEE.open("a", encoding="utf-8") as sink:
        sink.write(line + "\n")
        child = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT, text=True, bufsize=1)
        for out in child.stdout:
            if not quiet:
                print(out, end="", flush=True)
            sink.write(out)
        code = child.wait()
    if code:
        raise subprocess.CalledProcessError(code, cmd)


In [ ]:
# 2. Get the Lilly code — into scratch, NOT the Output directory
# Everything under /kaggle/working becomes version Output, and a git clone
# there floods `kaggle kernels output` with objects so the adapter zip never
# downloads (docs/kaggle-fail-stop.md, item 2). Only the zip belongs there.
SCRATCH = Path("/kaggle/temp") if Path("/kaggle/temp").is_dir() else Path("/tmp")
CLONE = SCRATCH / "Lilly"
subprocess.run(["rm", "-rf", str(CLONE)], check=True)
os.chdir(SCRATCH)
run("git", "clone", "-q", "https://github.com/ssaaffaakk/Lilly.git")
assert (CLONE / "training" / "train_translation.py").is_file(), "clone produced nothing"
os.chdir(CLONE)
print("working in", os.getcwd(), "— Output will hold only the zip")
from training.kaggle_offload import Offload
OFF = Offload("translation", os.environ.get("LILLY_RUN_ID", f"translation-{DIRECTION}-{ARM}"))
OFF.hardware(torch.cuda.get_device_name(0))


In [ ]:
# 3. Install what we need (~2 min)
# The versions come out of the repo's own requirements.txt rather than being
# copied here. A second hand-kept list is how the speech run died: peft was
# pinned in requirements.txt and simply missing from that notebook's copy, so
# Kaggle's own much newer peft was used and its torchao dispatcher raised on the
# first get_peft_model() call — after a 3 GB download.
NEEDED = ["transformers", "peft", "accelerate", "sacrebleu", "sentencepiece",
          "sacremoses", "ctranslate2"]  # ctranslate2: the forward build reads it for back-translation
pins = {}
for line in Path("requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.split("#")[0].strip()
    if "==" in line:
        pins[line.split("==")[0].strip().lower()] = line
unpinned = [n for n in NEEDED if n not in pins]
print("pinned here:", [pins[n] for n in NEEDED if n in pins])
print("no pin in requirements.txt, taking latest:", unpinned or "none")
run(sys.executable, "-m", "pip", "install", "-q", *[pins.get(n, n) for n in NEEDED])


In [ ]:
# 3b. Prove this machine can actually train, before hours are spent finding out
# Two things must hold that no version number shows: peft must be able to build a
# LoRA layer on this image, and the card Kaggle handed us must actually run
# kernels. A previous run satisfied "a GPU is available" on a P100 whose sm_60
# the installed PyTorch does not support, and failed only once it tried to
# compute — after the download.
import torch, torch.nn as nn
from peft import LoraConfig, get_peft_model
import peft, transformers
print("peft", peft.__version__, "| transformers", transformers.__version__)

class Tiny(nn.Module):
    def __init__(self):
        super().__init__()
        self.q_proj = nn.Linear(32, 32)
    def forward(self, x):
        return self.q_proj(x)

tiny = get_peft_model(Tiny(), LoraConfig(r=4, target_modules=["q_proj"]))
try:
    tiny = tiny.cuda()
    tiny(torch.randn(4, 32, device="cuda")).sum().backward()
except RuntimeError as exc:
    raise SystemExit(
        f"The GPU cannot run this build of PyTorch ({exc}).\n"
        f"Card: {torch.cuda.get_device_name(0)}. The accelerator is requested by "
        f"name in scripts/kaggle_train.py — it must be one of Kaggle's own "
        f"(NvidiaTeslaT4, NvidiaTeslaP100, ...), because an unrecognised name is "
        f"discarded silently and the run lands on the default card.") from exc

lora = [n for n, prm in tiny.named_parameters() if "lora_" in n and prm.grad is not None]
assert lora, "peft built a LoRA layer but no gradient reached it"
print(f"LoRA trains on {torch.cuda.get_device_name(0)}: "
      f"{len(lora)} adapter tensors took a gradient")
del tiny
torch.cuda.empty_cache()


In [ ]:
# 4. Find the en-bs base, the shipped bs->en forward build, MaCoCu-bs, and the corpus
import glob, hashlib

def build_fingerprint(build):
    """Identical to scripts/kaggle_train.translator_build_fingerprint and
    evaluate_app.build_fingerprint: blake2b over the build's files, name then
    bytes, sorted, skipping built.json and dataset-metadata.json."""
    d = hashlib.blake2b(digest_size=16)
    names = sorted(f for f in os.listdir(build)
                   if os.path.isfile(os.path.join(build, f))
                   and f not in ("dataset-metadata.json", "built.json"))
    for name in names:
        d.update(name.encode("utf-8"))
        with open(os.path.join(build, name), "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                d.update(chunk)
    return d.hexdigest()

# The shipped bs->en forward build, identified by fingerprint, not by a filename.
FORWARD_DIR = None
for meta in glob.glob("/kaggle/input/**/built.json", recursive=True):
    d = os.path.dirname(meta)
    if os.path.exists(os.path.join(d, "model.bin")) and build_fingerprint(d) == FORWARD_FINGERPRINT:
        FORWARD_DIR = d
        break
assert FORWARD_DIR, (
    f"No bs->en forward build with fingerprint {FORWARD_FINGERPRINT} among the "
    "attached datasets. scripts/kaggle_train.py uploads it as "
    "<user>/lilly-translator-scored and attaches it automatically -- launch with "
    "`python3 scripts/kaggle_train.py translation-en-bs-backtrans`, not by hand.")
print("forward build (bs->en, back-translation):", FORWARD_DIR)

# The en-bs base to fine-tune: a Marian dir with source.spm that knows >>bos_Latn<<
# and is NOT the CT2 forward build above.
from transformers import AutoTokenizer
BASE_DIR = None
for spm in glob.glob("/kaggle/input/**/source.spm", recursive=True):
    d = os.path.dirname(spm)
    if d == FORWARD_DIR or not os.path.exists(os.path.join(d, "config.json")):
        continue
    try:
        _tok = AutoTokenizer.from_pretrained(d)
    except Exception:
        continue
    _ids = _tok.convert_tokens_to_ids([PATHS["tag"]])
    if _ids and _ids[0] not in (None, _tok.unk_token_id):
        BASE_DIR = d
        break
assert BASE_DIR, (
    f'No en-bs base that knows {PATHS["tag"]} among the attached datasets. '
    "It is uploaded as <user>/lilly-translate-en-bs-base.")
os.environ["LILLY_BASE"] = BASE_DIR
print("en-bs base:", BASE_DIR)

# MaCoCu-bs 1.0 (CC0, CLARIN.SI 11356/1808): the raw corpus is prevert XML
# (MaCoCu-bs-1.0.xml, ~7 GB) crawled from .ba, or a pre-cleaned .txt. Too large
# to stage from a laptop, so the owner creates <user>/lilly-macocu-bs on Kaggle
# directly from the CLARIN URL; the notebook extracts + Latin-filters it next.
_raw = [p for p in glob.glob("/kaggle/input/**/*", recursive=True)
        if p.lower().endswith((".xml", ".txt")) and os.path.getsize(p) > 5_000_000]
_raw.sort(key=os.path.getsize, reverse=True)
assert _raw, (
    "Attach MaCoCu-bs 1.0 as a Dataset input (<user>/lilly-macocu-bs). It is "
    "~GBs, so create it on Kaggle directly from the CLARIN URL (Datasets -> New "
    "Dataset -> Link/URL: MaCoCu-bs-1.0.xml.zip), not uploaded from the Mac.")
MACOCU_RAW = _raw[0]
print("MaCoCu-bs raw:", MACOCU_RAW, f"({os.path.getsize(MACOCU_RAW) / 1e9:.2f} GB)")

# The pinned extra corpus, same as every translation run.
_corpus = sorted(glob.glob("/kaggle/input/**/extra-train.tsv", recursive=True))
assert _corpus, "Attach the Lilly extra corpus (<user>/lilly-extra-corpus)."
CORPUS = _corpus[0]
print("pinned corpus:", CORPUS)

In [ ]:
# 5. Download and clean the Bosnian-English data (~12 min)
import shutil

def rows_in(path):
    with open(path, encoding="utf-8") as fh:
        return sum(1 for _ in fh)

run("python3", "data/scripts/download_data.py")
run("python3", "data/scripts/clean_data.py")

# WikiMatrix is web-mined and about a ninth of it is not a translation pair at
# all — the two sides are merely about the same subject. Three rules, each
# measured against SETIMES and TED2020 (aligned by construction, so whatever
# fires there is the false-positive floor: 0.14%), drop 21,178 of them. 132
# pairs were read by hand across three rounds to calibrate. Left in, those pairs
# teach the model to invent.
run("python3", "data/scripts/filter_train_data.py")

# data/clean/train.tsv means exactly one thing from here on: the filter's output.
# It is never appended to. The previous version of this cell concatenated the
# extra corpus onto it, so the file's contents depended on how far the notebook
# had run -- and build_training_mix.py, which reads train.tsv AND the extra file,
# counted every extra pair twice: 390,172 rows against an expected 351,889. The
# mix builder is the single place that combines the two.
filtered = rows_in("data/clean/train.tsv")
assert filtered == 313_612, (
    f"the filter produced {filtered:,} rows, expected 313,612. Every count in "
    f"build_training_mix.py was measured on that number.")

# 38,277 pairs the base model has never seen: wikimedia-v20260327 and the
# professionally translated NTREX-128, copied from the attached dataset.
Path("data/extra").mkdir(parents=True, exist_ok=True)
shutil.copyfile(CORPUS, "data/extra/extra-train.tsv")
extra = rows_in("data/extra/extra-train.tsv")
assert extra == 38_277, f"the pinned corpus has {extra:,} rows, expected 38,277"

# The benchmark the base model has never seen. Our own split came from the same
# corpora it was trained on, so a gain there would prove nothing on its own.
run("python3", "data/scripts/download_flores.py")

# ---- leakage, measured here rather than inherited --------------------------
# download_extra_data.py deduplicates against valid/test by WHOLE PAIR and
# against FLORES by EITHER SIDE. Not re-harvesting skips that, so both inputs
# are re-checked here, before anything trains on them.
#
# The counts below are what this corpus actually contains, measured, not assumed:
#
#   data/clean/train.tsv        313,612 rows   whole-pair 0   FLORES 0   one-side 8
#   data/extra/extra-train.tsv   38,277 rows   whole-pair 0   FLORES 0   one-side 1
#
# Whole-pair overlap is the one that would let the model memorise a held-out
# answer, and it is zero. FLORES is the set the pre-registered decision is
# measured on, and it is zero. The nine one-side rows are citation boilerplate
# sharing one side with one held-out row -- "Aftenposten (in Norwegian)." against
# test.tsv line 1348 is the extra corpus's only one -- and eight of the nine are
# in the corpus the shipped model already trained on. They are pinned at their
# measured counts instead of asserted to zero: asserting zero would be asserting
# something that was never true of this corpus, and pinning catches a real jump
# while leaving a known, quantified 0.0026% visible rather than hidden.
def norm(s):
    return " ".join(s.lower().split())

held_side, held_pair = set(), set()
for split in ("data/clean/valid.tsv", "data/clean/test.tsv"):
    with open(split, encoding="utf-8") as fh:
        for line in fh:
            parts = line.rstrip("\n").split("\t")
            if len(parts) == 3:
                held_side.add(norm(parts[1]))
                held_side.add(norm(parts[2]))
                held_pair.add((norm(parts[1]), norm(parts[2])))
flores = set()
for path in sorted(glob.glob("data/flores/*")):
    if os.path.isfile(path):
        with open(path, encoding="utf-8") as fh:
            for line in fh:
                flores.add(norm(line))

for path, expect_side in (("data/clean/train.tsv", 8),
                          ("data/extra/extra-train.tsv", 1)):
    side = pair = flo = 0
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            parts = line.rstrip("\n").split("\t")
            if len(parts) != 3:
                continue
            nb, ne = norm(parts[1]), norm(parts[2])
            if (nb, ne) in held_pair:
                pair += 1
            if nb in flores or ne in flores:
                flo += 1
            if nb in held_side or ne in held_side:
                side += 1
    print(f"  {path}: whole-pair {pair}, FLORES {flo}, one-side {side}")
    assert pair == 0, f"{path} shares {pair} whole pairs with valid/test"
    assert flo == 0, f"{path} shares {flo} lines with FLORES — the decision set"
    assert side == expect_side, (
        f"{path} one-side overlap moved: {side}, was {expect_side} when measured")

print(f"{filtered:,} filtered + {extra:,} pinned = {filtered + extra:,} pairs into the mix")


In [ ]:
# 5b. Rebuild the corpus so its length variance matches the benchmark's
# The fine-tuning's chrF2 cost is entirely its terseness, and the cause is not
# the corpus mean — that already matches FLORES to 0.3% — but its variance:
# WikiMatrix carries sd(log char ratio) 0.213 against FLORES's 0.129, so the
# model learns from a much wider spread of length ratios than it is asked to
# produce. This splits to one sentence per example inside a length band tilted
# slightly long, and holds 462 ntrex pairs out for validation.
# --direction: step 6 caps pairs at 128 tokens under the tokenizer this run
# trains with, fed the way the trainer feeds it. Without it the en-bs base's
# English source.spm was counting Bosnian text and 4,067 pairs "crossed" the
# cap (version 2, 7 September); the script now carries one column of counts
# per direction and refuses a direction it has not measured.
run("python3", "data/scripts/build_training_mix.py", "--direction", DIRECTION)
mix_rows = sum(1 for _ in open("data/clean/train-mix.tsv", encoding="utf-8"))
holdout = sum(1 for _ in open("data/clean/ntrex-holdout.tsv", encoding="utf-8"))
print(f"{mix_rows:,} training examples, {holdout:,} held out")
assert mix_rows > 340_000, f"only {mix_rows:,} examples — the mix build fell short"


In [ ]:
# 5c. Back-translate MaCoCu-bs and build the 1:1 training mix (Run B's one change)
import time, random, re as _re
from pathlib import Path

# (a0) Extract Latin sentences from MaCoCu-bs. The corpus is prevert XML crawled
# from .ba and MIXES Latin + Cyrillic (hbs_cyr, largely Republika Srpska). Run B's
# targets must be Latin ijekavian Bosnian -- the en->bs model writes bos_Latn --
# so keep the paragraph text and DROP any line with a Cyrillic character. A raw
# mix would teach the model the wrong script. (Sentence splitting is left to the
# back-translation Engine, which splits internally; overlong lines are dropped by
# prepare_backtrans_bs.py's token cap.)
_CYR = _re.compile("[\u0400-\u04FF]")
MACOCU = "/kaggle/temp/macocu-bs-latin.txt"
_kept = _dropped = 0
with open(MACOCU_RAW, encoding="utf-8", errors="ignore") as _src, \
     open(MACOCU, "w", encoding="utf-8") as _dst:
    for _line in _src:
        _s = _line.strip()
        if not _s or _s.startswith("<"):      # prevert tags; a plain .txt has no leading '<'
            continue
        if _CYR.search(_s):
            _dropped += 1
            continue
        _dst.write(_s + "\n")
        _kept += 1
assert _kept > 100_000, (
    f"only {_kept:,} Latin lines from MaCoCu-bs -- too few to sample 1:1 against "
    f"the mix; check the attached corpus (is it the MaCoCu-bs-1.0 XML?)")
print(f"MaCoCu-bs: kept {_kept:,} Latin lines, dropped {_dropped:,} Cyrillic/tag")

# (a) Sample, clean and HOLD OUT MaCoCu-bs. The holdout guarantees not one FLORES
# dev/devtest or bench target, and nothing already in the en-bs mix, can leak into
# training (scripts/prepare_backtrans_bs.py refuses an empty holdout by design).
BT_SRC = "/kaggle/temp/backtrans-bs.txt"
run("python3", "scripts/prepare_backtrans_bs.py",
    "--input", MACOCU,
    "--holdout", "data/flores/dev.bs", "data/flores/devtest.bs",
    "--bench", "bench/cases.tsv",
    "--parallel", "data/clean/train-mix.tsv",
    "--n", str(BACKTRANS_N),
    "--out", BT_SRC,
    "--report", "/kaggle/working/backtrans-report.json")
bs_lines = [l.rstrip("\n") for l in open(BT_SRC, encoding="utf-8") if l.strip()]
assert bs_lines, "prepare_backtrans_bs.py produced no sentences -- nothing to back-translate"
print(f"{len(bs_lines):,} held-out MaCoCu-bs sentences to back-translate")

# (b) Back-translate bs->en with the SHIPPED forward build, through the product
# path (app.translate.Engine): synthetic English source, real Bosnian target.
import sys as _sys
_sys.path.insert(0, os.getcwd())
from app.translate import Engine
eng = Engine(directory=Path(FORWARD_DIR), direction="bs-en")
assert eng.device == "cuda", \
    "the forward build is not on the GPU -- 1M sentences on CPU misses the wall"

pairs = []                       # (real_bs, synthetic_en)
t0 = time.time()
for i, bs in enumerate(bs_lines):
    en = eng.translate(bs, truncate=True)
    if en.strip():
        pairs.append((bs, en.strip()))
    if (i + 1) % 5000 == 0:
        rate = (i + 1) / (time.time() - t0)
        print(f"  {i + 1:,}/{len(bs_lines):,}  ({rate:.1f}/s)", flush=True)
    if time.time() - t0 > MAX_BT_SECONDS:
        raise SystemExit(
            f"back-translation hit the {MAX_BT_SECONDS / 3600:.1f}h cap at "
            f"{i + 1:,}/{len(bs_lines):,} sentences. This run does not "
            f"under-produce silently: reduce BACKTRANS_N so generation fits one "
            f"session, or split generation into its own Kaggle run (owner's call). "
            f"Not shipping a partial mix as the pre-registered 1:1 run.")
assert len(pairs) > 0.9 * len(bs_lines), (
    f"only {len(pairs):,} of {len(bs_lines):,} back-translations were non-empty -- "
    f"the forward build is emitting blanks, not a corpus")
print(f"{len(pairs):,} synthetic en->bs pairs in {(time.time() - t0) / 60:.0f} min")

# (c) Assemble the 1:1 mix (pre-registration: 1:1 by sentence count, real
# UP-SAMPLED if smaller). Same 3-column schema train_translation.py reads:
# `corpus \t bosnian \t english`. The trainer adds >>bos_Latn<< to the source
# itself, so the tag is NOT written here.
real = [l.rstrip("\n") for l in open("data/clean/train-mix.tsv", encoding="utf-8") if l.strip()]
synth = [f"backtrans-macocu\t{bs}\t{en}" for bs, en in pairs]
rng = random.Random(20260914)
if len(real) < len(synth):
    real_up = real * (len(synth) // len(real) + 1)
    rng.shuffle(real_up)
    real_up = real_up[:len(synth)]
else:
    real_up = real[:len(synth)]
mixed = real_up + synth
rng.shuffle(mixed)
MIX = "data/clean/train-mix-backtrans.tsv"
with open(MIX, "w", encoding="utf-8") as fh:
    fh.write("\n".join(mixed) + "\n")
bad = [r for r in mixed if len(r.split("\t")) != 3]
assert not bad, f"{len(bad)} rows are not corpus\\tbs\\ten -- the trainer would skip them"
print(f"1:1 mix: {len(real_up):,} real (up-sampled) + {len(synth):,} synthetic = "
      f"{len(mixed):,} training examples -> {MIX}")

In [ ]:
# 6. Quick pipeline check (~3 min) — a toy run, just to prove everything works.
# It writes to models/quicktest-adapter, never to the real one.
run("python3", "-u", "training/train_translation.py", "--direction", DIRECTION,
    "--quick-test")

In [ ]:
# 7. THE REAL TRAINING (~3-4 hours) — the shipped en-bs LoRA recipe on the Run B
# mix. Two epochs, the 462 held-out ntrex pairs as validation, same seed as the
# shipped adapter; only --data differs from the shipped run.
common = ["--direction", DIRECTION,
          "--data", "data/clean/train-mix-backtrans.tsv",
          "--valid", "data/clean/ntrex-holdout.tsv",
          "--valid-limit", "0",
          "--epochs", "2"]
run("python3", "-u", "training/train_translation.py", *common)
assert Path(PATHS["adapter"] + "/adapter_config.json").is_file(), \
    f'no adapter was written to {PATHS["adapter"]}'

In [ ]:
# 8. In-run sanity read (~20 min). The DECIDING numbers are the four pre-registered
# bars on the SERVED path, scored AT HOME (training/PREREGISTRATION.md, "Run B"):
# chrF2 > 61.55 (CI excludes 0), BLEU CI-low >= -0.30, form rate >= 99.0%, label
# gap >= +15. evaluate.py here is the sanity read, not the gate.
run("python3", "-u", "training/evaluate.py", "--direction", DIRECTION,
    "--adapter", PATHS["adapter"])
print(Path(PATHS["results"]).read_text())
print("\nRun B deciding numbers are scored at home on the served path:")
print("  unzip lilly-adapter-en-bs-backtrans.zip to models/lilly/adapter-en-bs-backtrans/, then:")
print("  python3 scripts/build_translator.py --direction en-bs \\")
print("      --adapter models/lilly/adapter-en-bs-backtrans \\")
print("      --dest models/lilly/translator-en-bs-backtrans")
print("  python3 training/evaluate_app.py --direction en-bs --tuned models/lilly/translator-en-bs-backtrans")
print("  python3 training/bosnian_form_rate.py   # the form rate + label gap guardrails")

In [ ]:
# 9. Package the adapter so it survives the run. The tee is scanned first: a
# trainer that printed its own death and still returned 0 is a hole, not a result
# (fail-stop). The name carries the run so it cannot be unzipped over the shipped
# en-bs adapter.
OFF.check_trainproof()
assert Path(PATHS["results"]).is_file()
name = f'{PATHS["zip"]}.zip'
run("zip", "-qr", f"/kaggle/working/{name}", PATHS["adapter"], PATHS["results"])
out, floor = Path(f"/kaggle/working/{name}"), 1_000_000
size = out.stat().st_size
assert size > floor, f"{out.name} is only {size} bytes -- the save did not happen"
print(f"{out.name} — {size / 1048576:.1f} MB, in the Output tab when this finishes")
OFF.finish(ARM, [str(out), "/kaggle/working/stdout.txt",
                 "/kaggle/working/experiment_log.json",
                 "/kaggle/working/backtrans-report.json"])

**Done.** Download `lilly-adapter-en-bs-backtrans.zip` and `backtrans-report.json`
from the **Output** tab. The four pre-registered bars are scored at home on the
served path (cell 8 prints the commands); Run B ships only if all four hold.